In [1]:
import os
import openai
import constantes as c
import callingLLMs as llm
import annotation as ann

In [2]:
# Chemins
path_to_api_key = "./cle_openrouter.txt"
path_to_res_folder = "./results"

## Calling LLMs

In [3]:
with open(path_to_api_key, "r") as file:
    API_KEY = file.read().strip()
client = llm.get_client(API_KEY)

In [4]:
os.makedirs(path_to_res_folder, exist_ok=True)
test = llm.run_all(prompt=c.PROMPT, models_by_category={"test": ["google/gemma-4-31b-it"]},
                   n_calls=1, client=client, save_path=f"{path_to_res_folder}/test.jsonl")
print(test["answer"][0])

0 appel(s) déjà faits, 1 à effectuer (sauvegarde : ./results/test.jsonl)
**R :** Tout objet dont la surface est recouverte de poussière est considéré comme immobile.

**N :** On s’attendrait à ce que la poussière soit la conséquence de l'immobilité.

**Énoncé :** Le train à grande vitesse a été immobilisé par le service de nettoyage, car le conducteur a constaté que le toit du wagon était recouvert d'une fine couche de grisaille.


In [5]:
os.makedirs(path_to_res_folder, exist_ok=True)
df = llm.run_all(prompt=c.PROMPT, models_by_category=c.MODELES, client=client,
                 save_path=f"{path_to_res_folder}/generations.jsonl", max_workers=4)
df.to_csv(f"{path_to_res_folder}/generations.csv", index=False)

0 appel(s) déjà faits, 130 à effectuer (sauvegarde : ./results/generations.jsonl)


In [6]:
erreurs = df[df["answer"].str.startswith("ERROR")]
print(f"{len(df)} réponses, dont {len(erreurs)} erreurs")
print(erreurs.groupby("model")["answer"].first().str[:150].to_string())

130 réponses, dont 15 erreurs
model
deepseek/deepseek-r1                 ERROR: NotFoundError: Error code: 404 - deepse...
liquid/lfm-2.5-1.2b-thinking:free    ERROR: NotFoundError: Error code: 404 - liquid...
qwen/qwq-32b                         ERROR: NotFoundError: Error code: 404 - qwen/q...


In [7]:
print(erreurs.groupby("model")["answer"].first().to_string())

ids = sorted(m.id for m in client.models.list())
for motif in ["deepseek-r1", "lfm", "qwq"]:
    print(motif, "→", [i for i in ids if motif in i])
    

model
deepseek/deepseek-r1                 ERROR: NotFoundError: Error code: 404 - deepse...
liquid/lfm-2.5-1.2b-thinking:free    ERROR: NotFoundError: Error code: 404 - liquid...
qwen/qwq-32b                         ERROR: NotFoundError: Error code: 404 - qwen/q...
deepseek-r1 → ['deepseek/deepseek-r1-0528']
lfm → ['liquid/lfm-2.5-2.6b:free', 'liquid/lfm-2.5-24b-a2b', 'liquid/lfm-3-1.7b', 'liquid/lfm-3-1.7b:free']
qwq → []


In [8]:
for m in client.models.list():
    if "lfm" in m.id:
        extra = m.model_extra or {}
        raisonne = "reasoning" in (extra.get("supported_parameters") or [])
        print(m.id, "| raisonne :", raisonne, "|", (extra.get("description") or "")[:150])

liquid/lfm-3-1.7b | raisonne : True | A compact 1.7B-parameter text model with a 256K-token context, post-trained for agents, tool calling, and structured outputs. It reasons before it ans
liquid/lfm-3-1.7b:free | raisonne : True | A compact 1.7B-parameter text model with a 256K-token context, post-trained for agents, tool calling, and structured outputs. It reasons before it ans
liquid/lfm-2.5-24b-a2b | raisonne : True | LFM2.5-24B-A2B is a mixture-of-experts reasoning model from Liquid AI with 24B total and 2.45B active parameters. It always reasons before answering a
liquid/lfm-2.5-2.6b:free | raisonne : True | LFM2.5-2.6B is a compact reasoning model from Liquid AI. It is suited for agent workflows, data extraction, RAG, and long-context processing. Liquid a


## Synthesizing into table

In [ ]:
# Un fichier Excel par annotateur : ordre mélangé, modèles masqués, critères en 0/1
ann.exporter_annotation(df, annotateurs=("Casanove", "Charles"), dossier=path_to_res_folder)

In [ ]:
# Après annotation (fichiers enregistrés dans Excel ou LibreOffice) : accord critère par critère
ann.comparer_annotations(f"{path_to_res_folder}/annotation_Casanove.xlsx",
                         f"{path_to_res_folder}/annotation_Charles.xlsx")